# bordeus — valutazione RAG e confronto system prompt

Notebook per valutare la qualità del retrieval e confrontare varianti di
system prompt sullo stesso vector store che usa il bot in produzione —
senza dover passare da Telegram per ogni iterazione.

Riusa il codice reale del bot (`bordeus_bot.rag`: `QUERY_INSTRUCTION`,
`vocabolario_filter`, `SYSTEM_PROMPT_TEMPLATE`, `answer_question`) e lo
strumento reale (`bordeus_bot.calendario.make_tool`), non li duplica.

**Un solo percorso di generazione, quello vero.** Ogni risposta di questo
notebook — comprese quelle del confronto fra prompt — passa da
`answer_question` con gli strumenti attivi. Una versione precedente
reimplementava la generazione a mano per poter scambiare il prompt, ma
quella copia non faceva tool calling: confrontava prompt su un percorso
che il bot non usa più, e una variante poteva sembrare migliore solo
perché il giorno di raccolta non veniva mai chiesto. Ora il prompt si
scambia con il parametro `system_prompt_template` di `answer_question`,
e tutto il resto resta identico alla produzione.

**Un solo passaggio di retrieval** (vocabolario/guide): il calendario non
è più nel vector store. Le date vivono in `raccolta_date` e il modello le
ottiene chiamando `trova_prossima_raccolta`, a cui passa il **materiale**
dell'oggetto — quindi la parte "quando passano?" si valuta guardando se e
come il modello chiama lo strumento, non ispezionando i chunk recuperati.

**Il modello deve supportare il tool calling.** Verificato: `gemma3` non
lo supporta in Ollama, nemmeno nella variante 27b; `gemma4` sì.

**Log TRACE**: la cella di configurazione accende il livello `TRACE` su
`bordeus_bot`, che stampa il system prompt assemblato, i chunk
recuperati con la loro fonte, e gli argomenti/risultati di ogni chiamata
allo strumento. È il modo più diretto per capire *perché* una variante
si comporta diversamente da un'altra. Mettilo a `INFO` se il rumore
diventa eccessivo.

In [ ]:
import os
import sys

sys.path.insert(0, "../../common/src")
sys.path.insert(0, "../src")

from dotenv import load_dotenv

load_dotenv("../.env")

import pandas as pd
from bordeus_bot import calendario, i18n, identify
from bordeus_bot.rag import (
    QUERY_INSTRUCTION,
    SYSTEM_PROMPT_TEMPLATE,
    answer_question,
    vocabolario_filter,
)
from bordeus_common.embed import get_embeddings
from bordeus_common.log import set_level, setup_logging
from bordeus_common.vectorstore import get_vectorstore
from langchain_ollama import ChatOllama

setup_logging("INFO")
pd.set_option("display.max_colwidth", None)

## Configurazione

`area_id` deve corrispondere a un'area già ingerita. `comune_id`
è opzionale: lascialo vuoto (`""`) per vedere solo il contenuto
condiviso dell'area, oppure imposta un comune reale per includere anche
il suo contenuto specifico (es. un calendario) — vedi
`comune_filter` in `bot/rag.py`.

In [ ]:
database_url = os.environ["DATABASE_URL"]

area_id = "sub-ato-e"      # cambia con un'area realmente ingerita
comune_id = "donnas"       # id di un comune reale dell'area; "" = solo contenuto condiviso
hamlet = ""                # frazione dell'utente simulato: "albard", "crous", ... oppure ""

ollama_base_url = os.environ.get("OLLAMA_BASE_URL", "http://localhost:11434")
# Deve supportare il tool calling, altrimenti il giorno di raccolta non
# viene mai richiesto e il confronto fra prompt perde la metà che conta.
ollama_model = os.environ.get("OLLAMA_MODEL", "gemma4:latest")

# Lingua in cui il bot deve rispondere (uno tra "it", "fr", "en", "es",
# "de" — vedi bordeus_bot.i18n.SUPPORTED_LANGUAGES). Il vero bot la
# ricava da update.effective_user.language_code; qui la impostiamo a
# mano per confrontare le risposte nella stessa lingua.
test_language = "it"

# TRACE solo su bordeus_bot: mostra il system prompt assemblato, i chunk
# recuperati e le chiamate allo strumento, senza il rumore delle
# librerie. Metti "INFO" per spegnerlo.
set_level("bordeus_bot", "TRACE")

embeddings = get_embeddings(query_instruction=QUERY_INSTRUCTION)
vectorstore = get_vectorstore(database_url, area_id, embeddings)
llm = ChatOllama(model=ollama_model, base_url=ollama_base_url, temperature=0.2)

# Lo stesso strumento che il bot costruisce per ogni richiesta, con
# comune e frazione legati in chiusura (vedi bordeus_bot.calendario).
# Ricostruiscilo se cambi comune_id o hamlet qui sopra.
tools = [calendario.make_tool(database_url, comune_id, hamlet=hamlet)] if comune_id else []

destinazione = f"{comune_id}/{hamlet}" if hamlet else (comune_id or "(nessun comune)")
print(f"Area: {area_id!r}  |  Destinazione: {destinazione}  |  Modello: {ollama_model!r}")
if tools:
    print(f"\nStrumento offerto al modello: {tools[0].name}")
    print(f"Argomenti: {set(tools[0].args)}")
    print(f"\n{tools[0].description}")
else:
    print("\nNessuno strumento: senza comune_id il giorno di raccolta non è valutabile.")

## Domande di prova

Placeholder generiche — sostituiscile con domande pertinenti ai
contenuti realmente ingeriti per la tua area (nomi di materiali/oggetti
che sai essere trattati nelle guide, più qualche domanda "trabocchetto"
fuori tema per verificare che il bot ammetta di non saperlo invece di
inventare). `comune_id` per riga è opzionale: se assente/`None`, usa
quello configurato sopra — utile per domande specifiche di un comune
diverso da quello di default (es. un calendario).

In [ ]:
eval_questions = [
    {"domanda": "Come smaltisco questo oggetto: tazza per caffè, ceramica gialla con manico?"},
    # {"domanda": "Come smaltisco una bottiglia di plastica?"},
    # {"domanda": "Dove butto le pile esaurite?"},
    # {"domanda": "Come devo smaltire dei vecchi vestiti?"},
    # {"domanda": "Dove butto una tazza in ceramica rotta?"},
    # {"domanda": "Come smaltisco farmaci scaduti?"},
    # # Carta contro cartone: la stessa domanda deve dare giorni DIVERSI a
    # # seconda della destinazione impostata sopra (Donnas capoluogo li
    # # raccoglie insieme, le sue frazioni separatamente). È il caso in cui
    # # si vede se il modello passa allo strumento il materiale giusto.
    # {"domanda": "Dove butto una scatola di cartone?"},
    # {"domanda": "Dove butto dei vecchi giornali?"},
    # # Oggetto tipicamente NON raccolto porta a porta (va all'ecocentro):
    # # un buon prompt non deve chiamare lo strumento, e soprattutto non
    # # deve citare nessun giorno di raccolta.
    # {"domanda": "Posso conferire un elettrodomestico rotto nel cassonetto normale?"},
    # # Domanda "trabocchetto", fuori tema: un buon system prompt deve
    # # ammettere di non saperlo, non inventare una risposta plausibile.
    # {"domanda": "Qual è la ricetta della polenta valdostana?"},
]

for q in eval_questions:
    q.setdefault("comune_id", comune_id)

print(f"{len(eval_questions)} domande di prova")

## Step 0 — Estrazione della descrizione dell'oggetto

Replica il primo passo del bot vero (`identify.identify_object_from_text`),
non salta direttamente al retrieval: in produzione la query sul vector
store non è mai la domanda intera dell'utente, è solo la descrizione
dell'oggetto estratta da essa — usare la frase intera introdurrebbe
rumore (saluti, formule di cortesia, la struttura della domanda) nella
ricerca per similarità. Una domanda "trabocchetto" fuori tema (nessun
oggetto menzionato) deve restituire `None`: quella domanda salta
retrieval e generazione, esattamente come nel bot vero — non è un
errore, è il comportamento corretto.

In [4]:
for q in eval_questions:
    descrizione = identify.identify_object_from_text(llm, q["domanda"])
    q["descrizione_oggetto"] = descrizione
    esito = repr(descrizione) if descrizione is not None else "None (nessun oggetto riconosciuto)"
    print(f"{q['domanda']!r}\n  -> {esito}\n")

2026-08-25 11:45:01,442 INFO httpx: HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


'Come smaltisco questo oggetto: tazza per caffè, ceramica gialla con manico?'
  -> 'tazza per caffè, ceramica gialla con manico'



## Step 1 — Qualità del retrieval

Replica il retrieval usato in produzione (`bot/rag.py`,
`retrieve_vocabolario`), compreso l'uso della sola **descrizione
dell'oggetto** (estratta allo Step 0) come query, non la domanda intera.

Un solo passaggio, non più due: il calendario non è nel vector store.
Le date vivono in `raccolta_date` e il modello le ottiene chiamando lo
strumento `trova_prossima_raccolta` — quindi il retrieval qui riguarda
solo il vocabolario/le guide, e la parte "quando passano?" si valuta
allo Step 3, guardando se il modello chiama davvero lo strumento e con
quale categoria.

In [ ]:
def show_retrieval(descrizione_oggetto: str, comune_id_domanda: str, k_vocabolario: int = 6) -> None:
    """Ispeziona il retrieval con le distanze, che answer_question non
    espone. Serve a capire se una risposta sbagliata viene da un
    retrieval sbagliato o dal prompt: sono due problemi diversi con due
    correzioni diverse."""
    print(f"Descrizione oggetto: {descrizione_oggetto!r}")
    print(f"Filtrato per comune: {comune_id_domanda!r}\n")

    risultati = vectorstore.similarity_search_with_score(
        descrizione_oggetto, k=k_vocabolario, filter=vocabolario_filter(comune_id_domanda)
    )
    for doc, distanza in risultati:
        print(f"  distanza={distanza:.4f}  [{doc.metadata.get('tipo')}]  {doc.metadata.get('source')}")
        print(f"    {doc.page_content}\n")


for q in eval_questions:
    if q["descrizione_oggetto"] is None:
        print(f"'{q['domanda']}' — nessun oggetto riconosciuto, salto il retrieval (comportamento atteso)")
        print("-" * 80)
        continue
    show_retrieval(q["descrizione_oggetto"], q["comune_id"])
    print("-" * 80)

## Step 2 — Varianti di system prompt

`"attuale"` è importato direttamente da `bot/rag.py`
(`SYSTEM_PROMPT_TEMPLATE`) — è il vero prompt in produzione, non una
copia che potrebbe disallinearsi. Le altre sono punti di partenza da
modificare liberamente: l'idea è generarne quante ne servono,
confrontarle sulle stesse domande, e copiare la vincitrice in
`bot/rag.py`.

Ogni variante deve accettare gli stessi placeholder che
`answer_question` passa a `.format()`: `{lingua}`, `{tool_name}`,
`{vocabolario_context}`. Uno in più fa fallire la generazione con
`KeyError`, uno in meno viene semplicemente ignorato.

Una variante che **non** menziona lo strumento è un esperimento
legittimo — serve a vedere quanto il modello lo usa di sua iniziativa —
ma aspettati che il giorno di raccolta sparisca dalle risposte.

In [ ]:
# Ogni variante riceve gli stessi placeholder di SYSTEM_PROMPT_TEMPLATE:
# {lingua}, {vocabolario_context}, {tool_name} — e vengono passate ad
# answer_question, cioè allo stesso percorso del bot, strumenti
# compresi. Un placeholder in più fa fallire .format() con KeyError.
#
# Il giorno di raccolta NON è contesto da mettere nel prompt: è il
# risultato di una chiamata a strumento. Le varianti possono cambiare
# come lo si chiede, non da dove arriva.
PROMPT_VARIANTS = {
    "attuale": SYSTEM_PROMPT_TEMPLATE,

    "piu_conciso": """\
Sei un assistente che rappresenta un'azienda di gestione rifiuti.
Rispondi ESCLUSIVAMENTE in base al contesto sottostante, non usare conoscenza generale sullo smaltimento rifiuti anche se pensi di saperla: le regole cambiano da comune a comune.
Rispondi SEMPRE in {lingua}, indipendentemente dalla lingua del contesto sottostante.
Rispondi in UNA frase, massimo due. Niente preamboli, niente ripetizioni della domanda.
Usa "Modalità di smaltimento" per il bidone/modalità giusta. Se l'oggetto è raccolto porta a porta, chiama {tool_name} e aggiungi il giorno in coda; altrimenti ometti il giorno.
Se non c'è abbastanza informazione, dillo in poche parole.

Modalità di smaltimento:
{vocabolario_context}
""",

    "few_shot": """\
Sei un assistente cordiale che rappresenta un'azienda di gestione rifiuti. Rispondi ESCLUSIVAMENTE in base al contesto sottostante, non usare conoscenza generale sullo smaltimento rifiuti anche se pensi di saperla: le regole cambiano da comune a comune. Rispondi SEMPRE in {lingua}, come negli esempi (gli esempi sono in italiano solo per mostrare lo stile, non la lingua da usare).

Per il giorno di raccolta usa SEMPRE lo strumento {tool_name}: non calcolarlo né inventarlo mai da solo.

Esempio 1
Domanda: Dove butto una bottiglia di vetro?
Risposta: Nel bidone del vetro (Bidone Verde), senza tappo. La prossima raccolta è il 05/09/2026.

Esempio 2
Domanda: Qual è la capitale della Francia?
Risposta: Non è una domanda sui rifiuti, non posso aiutarti con questo.

Ora rispondi tu, usando "Modalità di smaltimento". Se non conosci la risposta, dillo esplicitamente.

Modalità di smaltimento:
{vocabolario_context}
""",
}

print("Varianti definite:", list(PROMPT_VARIANTS.keys()))
print(f"Lingua di risposta per il confronto: {test_language!r} ({i18n.language_name(test_language)})")

## Step 3 — Esegui il confronto

Ogni variante passa da `answer_question` con gli strumenti attivi: lo
stesso identico percorso del bot, cambiando solo il testo del prompt. È
il motivo per cui questo notebook non ha più una funzione di generazione
propria — ne aveva una, senza tool calling, e confrontava prompt su un
percorso che il bot non usa.

Il retrieval viene rifatto dentro `answer_question` per ogni variante.
Costa qualche ricerca in più, ma è deterministico (stessa query, stesso
filtro, stesso `k`), quindi le differenze fra le risposte restano
attribuibili al prompt.

Con `len(eval_questions) * len(PROMPT_VARIANTS)` chiamate all'LLM, più le
chiamate allo strumento, può richiedere qualche minuto. I log TRACE
mostrano il prompt assemblato e ogni invocazione dello strumento.

In [ ]:
risultati_confronto = []
for q in eval_questions:
    if q["descrizione_oggetto"] is None:
        print(f"'{q['domanda']}' — nessun oggetto riconosciuto allo Step 0, salto: il bot vero si ferma qui")
        continue

    riga = {"domanda": q["domanda"]}
    for nome_variante, template in PROMPT_VARIANTS.items():
        riga[nome_variante] = answer_question(
            llm,
            vectorstore,
            q["comune_id"],
            q["descrizione_oggetto"],
            tools=tools,
            language_code=test_language,
            system_prompt_template=template,
        )
    risultati_confronto.append(riga)

df_confronto = pd.DataFrame(risultati_confronto).set_index("domanda")
print(f"\n{len(risultati_confronto)} domande x {len(PROMPT_VARIANTS)} varianti generate")

### Confronto leggibile, una domanda alla volta

Una tabella larga con colonne di testo lungo è scomoda da leggere in un
notebook — stampiamo invece un blocco per domanda, con tutte le
varianti sotto, una sopra l'altra.

In [ ]:
for domanda, riga in df_confronto.iterrows():
    print("=" * 80)
    print(f"DOMANDA: {domanda}\n")
    for nome_variante in PROMPT_VARIANTS:
        print(f"--- {nome_variante} ---")
        print(riga[nome_variante])
        print()

## Step 4 — Controlli automatici di base

Euristiche semplici, non un giudizio di qualità: servono a individuare
rapidamente le risposte chiaramente problematiche (vuote, che ripetono
la domanda, con placeholder non sostituiti, o in una lingua diversa da
quella attesa) prima di leggere tutto a mano. Un controllo passato non
garantisce che la risposta sia *corretta* — solo che non ha questi
problemi specifici.

In particolare **non** verificano la cosa più importante: che il giorno
di raccolta citato sia quello giusto. Quello si controlla nei log TRACE,
guardando con quale materiale è stato chiamato
`trova_prossima_raccolta` e cosa ha restituito — una risposta può essere
scorrevole, in italiano e della lunghezza giusta, e citare il giorno del
flusso sbagliato.

In [ ]:
_PAROLE_ITALIANE_COMUNI = {"il", "la", "di", "che", "per", "non", "un", "una", "sono", "puoi", "è", "e"}

# Placeholder che answer_question sostituisce: se ne compare uno nella
# risposta, la variante ne ha uno che .format() non conosceva, oppure il
# modello ha copiato il prompt nell'output.
_PLACEHOLDER = ("{lingua}", "{tool_name}", "{vocabolario_context}", "{context}")


def controlli_base(risposta: str, domanda: str) -> dict:
    parole = set(risposta.lower().replace(",", " ").replace(".", " ").split())
    return {
        "non_vuota": len(risposta.strip()) > 0,
        "non_eco_domanda": risposta.strip().lower() != domanda.strip().lower(),
        "niente_placeholder": not any(p in risposta for p in _PLACEHOLDER),
        "lunghezza_ragionevole": 5 <= len(risposta.split()) <= 200,
        "sembra_italiano": len(parole & _PAROLE_ITALIANE_COMUNI) > 0,
    }


righe_scorecard = []
for domanda, riga in df_confronto.iterrows():
    for nome_variante in PROMPT_VARIANTS:
        esiti = controlli_base(riga[nome_variante], domanda)
        righe_scorecard.append({
            "domanda": domanda,
            "variante": nome_variante,
            **esiti,
            "tutti_ok": all(esiti.values()),
        })

df_scorecard = pd.DataFrame(righe_scorecard)

print("Riepilogo per variante (quota di controlli superati su tutte le domande):\n")
print(df_scorecard.groupby("variante")["tutti_ok"].mean().sort_values(ascending=False))

print("\nRighe con almeno un controllo fallito (da leggere con attenzione):")
df_scorecard[~df_scorecard["tutti_ok"]]

## Estendere la valutazione

- **Più domande**: aggiungi voci a `eval_questions` — includi sempre
  qualche domanda fuori tema, è il modo più semplice per verificare che
  il prompt non "inventi" risposte, e almeno una coppia carta/cartone se
  l'area ha destinazioni che li dividono.
- **Più varianti**: aggiungi voci a `PROMPT_VARIANTS`. Devono usare solo
  i placeholder `{lingua}`, `{tool_name}`, `{vocabolario_context}`: uno
  in più fa fallire `.format()` con `KeyError`.
- **Cambia destinazione**: rimetti `comune_id`/`hamlet` nella cella di
  configurazione e ricostruisci `tools`. Con una frazione che ha uno
  schema di raccolta diverso, le stesse domande su carta e cartone
  devono dare giorni diversi: è la verifica più diretta che la
  risoluzione per materiale funzioni.
- **Un vero ground truth**: se hai risposte di riferimento scritte a
  mano, un passo naturale è confrontarle con quelle generate (a occhio,
  o con un LLM-as-judge — un secondo prompt che valuta la risposta
  contro il riferimento su una scala 1-5). Non implementato qui:
  richiede cura nel prompt di valutazione per essere affidabile, e un
  ground truth che questo notebook non presuppone.